# 🚀 DataMate: 1-Click Runner on Kaggle (Dual NVIDIA T4 GPUs)

This notebook provides a seamless, **100% automated 1-click execution** of the **DataMate** agent pipeline on **Kaggle's Free Dual Tesla T4 GPUs (32GB VRAM total)**.

### ⚡ Why this notebook is 100% Plug-and-Play:
- **No manual file upload needed**: Clones code directly from GitHub (`branch loc`).
- **Dual T4 Acceleration (`VLLM_TP=2`)**: Runs Tensor Parallelism across both T4 GPUs.
- **High-Speed Model Download**: Direct parallel download via `hf-transfer`.
- **Automatic Scoring**: Computes official Penalized Accuracy (PA) and generates `error_analysis.csv`.
- **1-Click Download**: Automatically zips results into `/kaggle/working/results_run12.zip` for instant download in the Kaggle sidebar.

---
> **⚠️ Required Kaggle Settings (Right Sidebar):**
> - **Accelerator:** `GPU T4 x 2`
> - **Internet:** `On`
> Then simply click **Run All**!


## 1. Hardware & GPU Acceleration Verification
Verifies that both NVIDIA Tesla T4 GPUs (16GB each) are detected and CUDA is active.

In [ ]:
import torch
import shutil

print("=" * 60)
print("🔍 HARDWARE & GPU VERIFICATION")
print("=" * 60)
!nvidia-smi

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"\n✅ Detected {gpu_count} NVIDIA GPU(s):")
    for i in range(gpu_count):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"   [GPU {i}] {name} - {vram:.1f} GB VRAM")
    assert gpu_count >= 1, "At least 1 GPU is required."
else:
    raise RuntimeError("❌ No CUDA GPU detected! Please enable 'GPU T4 x 2' in Kaggle settings.")

total, used, free = shutil.disk_usage(".")
print(f"💾 Free Disk Space: {free / (1024**3):.1f} GB out of {total / (1024**3):.1f} GB")
print("=" * 60)


## 2. Clone / Update DataMate Repository from GitHub
Fetches the latest Phase 11 code directly from branch `loc` using fast shallow clone.

In [ ]:
import os

REPO_URL = "https://github.com/vohoangloc-32/datamate.git"
BRANCH = "loc"
WORKDIR = "/kaggle/working/datamate"

if not os.path.exists(os.path.join(WORKDIR, "src/predict.py")):
    print(f"⬇️ Cloning DataMate repository ({BRANCH})...")
    !rm -rf {WORKDIR}
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {WORKDIR}
else:
    print(f"🔄 Pulling latest updates from branch {BRANCH}...")
    !cd {WORKDIR} && git fetch origin {BRANCH} --depth 1 && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

%cd {WORKDIR}
print("\n📌 Current Commit:")
!git log -1 --oneline


## 3. Install Dependencies & Verify Test Suite
Installs vLLM, acceleration dependencies, and executes the full unit test suite (201 tests).

In [ ]:
print("📦 Installing vLLM and DataMate requirements on Kaggle...")
!pip install -q vllm openpyxl xlrd tabulate pytest huggingface_hub pandas hf-transfer accelerate

print("\n🧪 Running full test suite (201 tests)...")
!pytest tests/ -q


## 4. Download Model Weights Directly from Hugging Face
Downloads `Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` directly to local NVMe storage.

In [ ]:
import os
import time
from huggingface_hub import snapshot_download

MODEL_REPO = "Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
LOCAL_MODEL_DIR = "/kaggle/working/model_cache/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

os.environ.pop("HF_ENDPOINT", None)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(f"⬇️ Downloading {MODEL_REPO} to {LOCAL_MODEL_DIR}...")
t0 = time.time()
snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=LOCAL_MODEL_DIR,
    local_dir_use_symlinks=False,
    max_workers=8
)
t_dl = time.time() - t0
print(f"✅ Model weights ready in {t_dl:.1f}s ({t_dl/60:.2f} min)!")


## 5. Run DataMate Prediction Pipeline (Dual T4 Optimized)
Runs inference using vLLM Tensor Parallelism (`VLLM_TP=2`) across both Tesla T4 GPUs.

In [ ]:
import os
import time
import torch

# Multi-GPU Tensor Parallelism across Dual T4 GPUs (32GB VRAM total)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
os.environ["VLLM_TP"] = "2" if gpu_count >= 2 else "1"
os.environ["VLLM_DTYPE"] = "auto"
os.environ["VLLM_MAX_MODEL_LEN"] = "16384"
os.environ["VLLM_MAX_NEW_TOKENS"] = "1024"

# Execution mode: 'train_unseen_500' | 'train_500' | 'public_test' | 'train_200'
MODE = "train_unseen_500"
CONFIDENCE_THRESHOLD = "0.45"

WORKDIR = "/kaggle/working/datamate"
if MODE == "public_test":
    DATA_DIR = os.path.join(WORKDIR, "data/public_test")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/public_test/questions.jsonl")
elif MODE == "train_unseen_500":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_unseen_500.jsonl")
elif MODE == "train_500":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_500.jsonl")
elif MODE == "train_200":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_200.jsonl")
else:
    raise ValueError(f"Unknown MODE: {MODE}")

OUTPUT_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("🚀 DATAMATE DUAL T4 INFERENCE ENGINE")
print("=" * 60)
print(f"GPUs Detected     : {gpu_count} (Tensor Parallel = {os.environ['VLLM_TP']})")
print(f"Execution Mode    : {MODE}")
print(f"Data Directory    : {DATA_DIR}")
print(f"Questions File    : {QUESTIONS_FILE}")
print(f"Output Directory  : {OUTPUT_DIR}")
print(f"Model Directory   : {LOCAL_MODEL_DIR}")
print(f"Confidence Thresh : {CONFIDENCE_THRESHOLD}")
print("=" * 60)

t_start = time.time()
!python src/predict.py \
    --data-dir {DATA_DIR} \
    --questions-file {QUESTIONS_FILE} \
    --output-dir {OUTPUT_DIR} \
    --model-dir {LOCAL_MODEL_DIR} \
    --confidence-threshold {CONFIDENCE_THRESHOLD} \
    --log-level INFO

elapsed = time.time() - t_start
print(f"\n🎉 Pipeline finished in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)!")


## 6. Compute Official Penalized Accuracy (PA) & Benchmark Results
Evaluates predictions against ground truth answers using the competition metric:
$$\text{PA} = \frac{\text{Correct} - \text{Wrong}}{\text{Total}}$$

In [ ]:
import os
import json
import pandas as pd
from src.evaluate import compute_pa

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.jsonl")
ANSWERS_PATH = os.path.join(DATA_DIR, "answers.jsonl")

if os.path.exists(SUBMISSION_PATH):
    submission_records = [json.loads(l) for l in open(SUBMISSION_PATH, encoding="utf-8-sig") if l.strip()]
    print(f"✅ Generated submission file: {SUBMISSION_PATH} with {len(submission_records)} records.")
    
    if os.path.exists(ANSWERS_PATH):
        preds = {r["id"]: r for r in submission_records}
        refs = {json.loads(l)["id"]: json.loads(l) for l in open(ANSWERS_PATH, encoding="utf-8-sig") if l.strip() and json.loads(l)["id"] in preds}
        metrics = compute_pa(preds, refs)
        total = metrics["total"]
        correct = metrics["correct"]
        wrong = metrics["wrong"]
        abstain = metrics["abstain"]
        pa = metrics["pa"]
        
        print("\n" + "=" * 60)
        print("🏆 DATAMATE OFFICIAL BENCHMARK SCORE (EVALUATED SUBSET)")
        print("=" * 60)
        print(f"Total Questions  : {total}")
        print(f"Correct (+1 pt)  : {correct} ({correct/total*100:.1f}%)")
        print(f"Wrong   (-1 pt)  : {wrong} ({wrong/total*100:.1f}%)")
        print(f"Abstained (0 pt) : {abstain} ({abstain/total*100:.1f}%)")
        print("-" * 60)
        print(f"🔥 Penalized Accuracy (PA): {pa:+.4f} ({pa*100:+.2f}%)")
        print("=" * 60)
        
        type_rows = []
        for t, s in sorted(metrics.get("per_type", {}).items()):
            tot = s["total"]
            cor = s["correct"]
            wro = s["wrong"]
            abs_ = s["abstain"]
            sub_pa = (cor - wro) / tot if tot > 0 else 0.0
            type_rows.append({
                "Type": t,
                "Total": tot,
                "Correct": cor,
                "Wrong": wro,
                "Abstain": abs_,
                "PA": f"{sub_pa:+.4f}",
                "Acc%": f"{cor/tot*100:.1f}%" if tot > 0 else "0.0%"
            })
        df_types = pd.DataFrame(type_rows)
        print(df_types.to_string(index=False))
        print("=" * 60)


## 7. Diagnostic Inspector: Error Analysis Breakdown
Generates `error_analysis.csv` and `abstain_analysis.csv` enriched with business ID and question text.

In [ ]:
import os
import json
import pandas as pd
from src.evaluate import score_answer

if os.path.exists(ANSWERS_PATH):
    questions_map = {}
    if os.path.exists(QUESTIONS_FILE):
        with open(QUESTIONS_FILE, "r", encoding="utf-8-sig") as f:
            for line in f:
                if line.strip():
                    q = json.loads(line)
                    questions_map[q["id"]] = q
                    
    gt_map = {}
    with open(ANSWERS_PATH, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                ans = json.loads(line)
                gt_map[ans["id"]] = ans

    sub_map = {}
    if os.path.exists(SUBMISSION_PATH):
        with open(SUBMISSION_PATH, "r", encoding="utf-8-sig") as f:
            for line in f:
                if line.strip():
                    sub = json.loads(line)
                    sub_map[sub["id"]] = sub

    errors = []
    abstains = []
    for q_id, gt in gt_map.items():
        if q_id not in sub_map:
            continue
        sub = sub_map[q_id]
        score = score_answer(sub, gt)
        q_meta = questions_map.get(q_id, {})
        
        row = {
            "id": q_id,
            "business_id": q_meta.get("business_id", ""),
            "question": q_meta.get("question", ""),
            "pred_type": sub.get("type"),
            "gt_type": gt.get("type"),
            "pred_val": sub.get("value"),
            "gt_val": gt.get("value"),
            "as_of_date": q_meta.get("as_of_date", "")
        }
        if score < 0:
            errors.append(row)
        elif score == 0:
            abstains.append(row)
            
    df_err = pd.DataFrame(errors)
    df_abs = pd.DataFrame(abstains)
    
    err_csv = os.path.join(OUTPUT_DIR, "error_analysis.csv")
    abs_csv = os.path.join(OUTPUT_DIR, "abstain_analysis.csv")
    df_err.to_csv(err_csv, index=False, encoding="utf-8-sig")
    df_abs.to_csv(abs_csv, index=False, encoding="utf-8-sig")
    print(f"📊 Saved error analysis report   : {err_csv} ({len(df_err)} errors)")
    print(f"📊 Saved abstain analysis report : {abs_csv} ({len(df_abs)} abstains)")


## 8. Package Results for 1-Click Download in Kaggle Sidebar
Compresses all outputs into `/kaggle/working/results_run12.zip` for instant download.

In [ ]:
import shutil

archive_path = "/kaggle/working/results_run12"
shutil.make_archive(archive_path, "zip", OUTPUT_DIR)
print(f"📦 Successfully created archive: {archive_path}.zip")
print("👉 Look at the Kaggle right sidebar under 'Output' to download results_run12.zip directly!")
